In [1]:
import pandas as pd

In [2]:
df=pd.read_csv('industrial_failure_dataset_global_lite.csv')

In [3]:
data=df.copy()

In [4]:
data.shape

(999, 15)

In [5]:
#shape after removing empty columns
data.isnull().sum()

event_id               0
timestamp              0
region                 0
area                   0
equipment              0
failure_name           0
failure_category       0
severity               0
root_cause             0
root_cause_category    0
impact_min             0
mttr_min               0
mtbf_hours             1
year                   0
month                  0
dtype: int64

In [6]:
data=data.dropna(how='all')

In [7]:
data.shape

(999, 15)

In [8]:
data.columns

Index(['event_id', 'timestamp', 'region', 'area', 'equipment', 'failure_name',
       'failure_category', 'severity', 'root_cause', 'root_cause_category',
       'impact_min', 'mttr_min', 'mtbf_hours', 'year', 'month'],
      dtype='object')

Converting timestamp column into data and time type

In [9]:
data['timestamp'] = pd.to_datetime(data['timestamp'], dayfirst=True)

In [10]:
data['hour'] = data['timestamp'].dt.hour
data['dayofweek'] = data['timestamp'].dt.dayofweek

In [11]:
data.columns

Index(['event_id', 'timestamp', 'region', 'area', 'equipment', 'failure_name',
       'failure_category', 'severity', 'root_cause', 'root_cause_category',
       'impact_min', 'mttr_min', 'mtbf_hours', 'year', 'month', 'hour',
       'dayofweek'],
      dtype='object')

Encode the failure_name column using lable encoder

In [12]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data['failure_name_enc'] = le.fit_transform(data['failure_name'])

In [13]:
data.columns

Index(['event_id', 'timestamp', 'region', 'area', 'equipment', 'failure_name',
       'failure_category', 'severity', 'root_cause', 'root_cause_category',
       'impact_min', 'mttr_min', 'mtbf_hours', 'year', 'month', 'hour',
       'dayofweek', 'failure_name_enc'],
      dtype='object')

One hot encoding for failure_category

In [14]:
data = pd.get_dummies(
    data,
    columns=['failure_category'],
    prefix='failure_category'
)

In [15]:
data.columns

Index(['event_id', 'timestamp', 'region', 'area', 'equipment', 'failure_name',
       'severity', 'root_cause', 'root_cause_category', 'impact_min',
       'mttr_min', 'mtbf_hours', 'year', 'month', 'hour', 'dayofweek',
       'failure_name_enc', 'failure_category_Electrical',
       'failure_category_Idle/External',
       'failure_category_Logistics/Organization',
       'failure_category_Mechanical', 'failure_category_Other'],
      dtype='object')

**Handling outliers of impact_min column**

In [16]:
data = data[data['impact_min'] <= 500]

**Appling log to impact_min column to reduce the skewness**

In [17]:
import numpy as np
data['impact_min_log'] = np.log1p(data['impact_min'])

Dropping the unnecessary columns

In [18]:
cols_to_drop = [
    'event_id',
    'timestamp',
    'region',
    'area',
    'equipment',
    'failure_name',
    'root_cause',
    'root_cause_category',
    'impact_min',
    'mttr_min',
    'year',
    'severity'
]


In [19]:
data.drop(columns=cols_to_drop, inplace=True,errors='ignore')

In [20]:
data.columns

Index(['mtbf_hours', 'month', 'hour', 'dayofweek', 'failure_name_enc',
       'failure_category_Electrical', 'failure_category_Idle/External',
       'failure_category_Logistics/Organization',
       'failure_category_Mechanical', 'failure_category_Other',
       'impact_min_log'],
      dtype='object')

In [21]:
data.isnull().sum()

mtbf_hours                                 0
month                                      0
hour                                       0
dayofweek                                  0
failure_name_enc                           0
failure_category_Electrical                0
failure_category_Idle/External             0
failure_category_Logistics/Organization    0
failure_category_Mechanical                0
failure_category_Other                     0
impact_min_log                             0
dtype: int64

**Splitting data among X and u variables**

In [22]:
X = data.drop('impact_min_log', axis=1)
y = data['impact_min_log']

**Splitting X and y into train and test**

In [23]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

Scaling the mtbf_hours column so that model can learn it easily

In [24]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

x_train[['mtbf_hours']] = scaler.fit_transform(x_train[['mtbf_hours']])
x_test[['mtbf_hours']] = scaler.transform(x_test[['mtbf_hours']])

Common code for Evaluting Different models

In [25]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_model(model, X_train, X_test, y_train, y_test, name):
    
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    
    # convert back to actual
    y_test_actual = np.expm1(y_test)
    y_pred_actual = np.expm1(y_pred)
    
    mae = mean_absolute_error(y_test_actual, y_pred_actual)
    rmse = np.sqrt(mean_squared_error(y_test_actual, y_pred_actual))
    r2 = r2_score(y_test_actual, y_pred_actual)
    
    print(f"{name}")
    print("MAE:", mae)
    print("RMSE:", rmse)
    print("R2:", r2)
    print("-"*40)
    
    return mae, rmse, r2

**Linear Regression**

In [26]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
evaluate_model(lr, x_train, x_test, y_train, y_test, "Linear Regression")

Linear Regression
MAE: 27.956108804816434
RMSE: 90.85164977406868
R2: 0.35860674875455323
----------------------------------------


(27.956108804816434, 90.85164977406868, 0.35860674875455323)

**Decision Tree Regressor**

In [27]:
from sklearn.tree import DecisionTreeRegressor

for depth in [3, 5, 10]:
    dt = DecisionTreeRegressor(max_depth=depth, random_state=42)
    evaluate_model(dt, x_train, x_test, y_train, y_test, f"Decision Tree depth={depth}")

Decision Tree depth=3
MAE: 4.639094327908616
RMSE: 11.397118373915657
R2: 0.9899063354398335
----------------------------------------
Decision Tree depth=5
MAE: 4.582758764942779
RMSE: 11.38157396670327
R2: 0.9899338499448006
----------------------------------------
Decision Tree depth=10
MAE: 4.544098530251271
RMSE: 11.364595610214945
R2: 0.9899638597150447
----------------------------------------


Decision tree with the  depth of 10 is better model

**Random Forest**

In [28]:
from sklearn.ensemble import RandomForestRegressor

for n in [100, 200, 300]:
    for depth in [5, 10]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            min_samples_split=5,
            min_samples_leaf=2,
            random_state=42
        )
        evaluate_model(rf, x_train, x_test, y_train, y_test,
                       f"RF n={n}, depth={depth}")

RF n=100, depth=5
MAE: 7.9344386163090155
RMSE: 29.54719108714176
R2: 0.9321591701801786
----------------------------------------
RF n=100, depth=10
MAE: 7.532202385730218
RMSE: 29.52502318040987
R2: 0.9322609277404131
----------------------------------------
RF n=200, depth=5
MAE: 7.937013666209865
RMSE: 28.77949725731716
R2: 0.935638648371668
----------------------------------------
RF n=200, depth=10
MAE: 7.5879952074128685
RMSE: 28.69861675240618
R2: 0.9359998960902777
----------------------------------------
RF n=300, depth=5
MAE: 7.771737947045519
RMSE: 27.75846510844142
R2: 0.9401244319737356
----------------------------------------
RF n=300, depth=10
MAE: 7.4457111747196665
RMSE: 27.646070439384076
R2: 0.9406083254658164
----------------------------------------


In [29]:
from sklearn.ensemble import GradientBoostingRegressor

for lr_rate in [0.05, 0.1]:
    gb = GradientBoostingRegressor(
        n_estimators=200,
        learning_rate=lr_rate,
        max_depth=3
    )
    evaluate_model(gb, x_train, x_test, y_train, y_test,
                   f"GB lr={lr_rate}")

GB lr=0.05
MAE: 7.6346407753458445
RMSE: 20.451068647033022
R2: 0.9674994460857969
----------------------------------------
GB lr=0.1
MAE: 7.5060520582284775
RMSE: 20.15099731286605
R2: 0.9684461875352539
----------------------------------------


gradient boosting with learning_rate of 0.1 is better than with 0.05 and random forest so i choose lr with 0.1

In [30]:
from sklearn.neighbors import KNeighborsRegressor

for k in [3, 5, 7]:
    knn = KNeighborsRegressor(n_neighbors=k)
    evaluate_model(knn, x_train, x_test, y_train, y_test,
                   f"KNN k={k}")

KNN k=3
MAE: 8.343609772047046
RMSE: 45.84490604912158
R2: 0.8366794160945303
----------------------------------------
KNN k=5
MAE: 13.92441310053644
RMSE: 63.39490402440483
R2: 0.6877032625471509
----------------------------------------
KNN k=7
MAE: 17.33944408666284
RMSE: 72.62047426570922
R2: 0.5901953221071548
----------------------------------------


KNN is not that good with this data, so i do not use this model 


In [31]:
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeRegressor
import numpy as np

In [32]:
dt = DecisionTreeRegressor(max_depth=5, random_state=42)

scoring = {
    'mae': 'neg_mean_absolute_error',
    'rmse': 'neg_root_mean_squared_error',
    'r2': 'r2'
}

scores = cross_validate(
    dt,
    x_train,
    y_train,
    scoring=scoring,
    cv=5,
    return_train_score=False
)

# convert values
mae = -scores['test_mae']
rmse = -scores['test_rmse']
r2 = scores['test_r2']

print("Decision Tree Results:")
print("MAE:", mae)
print("Avg MAE:", np.mean(mae))
print("RMSE:", rmse)
print("Avg RMSE:", np.mean(rmse))
print("R2:", r2)
print("Avg R2:", np.mean(r2))

Decision Tree Results:
MAE: [0.59050613 0.63928137 0.67915247 0.72270454 0.54057287]
Avg MAE: 0.6344434770756804
RMSE: [0.78242312 0.88867674 0.9256233  1.20716619 0.78001137]
Avg RMSE: 0.9167801437346463
R2: [0.60686421 0.55831085 0.52801876 0.24067874 0.63875021]
Avg R2: 0.514524553193844


In [33]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=10, random_state=42)

scores = cross_validate(
    rf,
    x_train,
    y_train,
    scoring=scoring,
    cv=5
)

mae = -scores['test_mae']
rmse = -scores['test_rmse']
r2 = scores['test_r2']

print("Random Forest Avg MAE:", np.mean(mae))
print("Random Forest Avg RMSE:", np.mean(rmse))
print("Random Forest Avg R2:", np.mean(r2))

Random Forest Avg MAE: 0.5864387485521003
Random Forest Avg RMSE: 0.8167977016903232
Random Forest Avg R2: 0.6101574630968685


In [34]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsRegressor

knn_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('knn', KNeighborsRegressor(n_neighbors=5))
])

scores = cross_validate(
    knn_pipeline,
    x_train,
    y_train,
    scoring=scoring,
    cv=5
)

mae = -scores['test_mae']
rmse = -scores['test_rmse']
r2 = scores['test_r2']

print("KNN Avg MAE:", np.mean(mae))
print("KNN Avg RMSE:", np.mean(rmse))
print("KNN Avg R2:", np.mean(r2))

KNN Avg MAE: 0.7091573690334011
KNN Avg RMSE: 1.053356280975286
KNN Avg R2: 0.3554379362285788


In [35]:
import numpy as np
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
def evaluate(name, model, X_train, y_train, X_test, y_test):
    model.fit(X_train, y_train)
    preds_log = model.predict(X_test)
    
    # metrics on log scale (what the model was trained/optimized on)
    r2_log = r2_score(y_test, preds_log)
    mae_log = mean_absolute_error(y_test, preds_log)
    rmse_log = np.sqrt(mean_squared_error(y_test, preds_log))
    
    # convert both predictions and actuals back to real minutes
    preds_actual = np.expm1(preds_log)
    y_test_actual = np.expm1(y_test)
    
    # metrics on original scale (what actually matters for interpretation)
    r2_actual = r2_score(y_test_actual, preds_actual)
    mae_actual = mean_absolute_error(y_test_actual, preds_actual)
    rmse_actual = np.sqrt(mean_squared_error(y_test_actual, preds_actual))
    
    print(f"{name}")
    print(f"  [log scale]      R2: {r2_log:.4f}  MAE: {mae_log:.4f}  RMSE: {rmse_log:.4f}")
    print(f"  [minutes scale]  R2: {r2_actual:.4f}  MAE: {mae_actual:.2f} min  RMSE: {rmse_actual:.2f} min")
    print()
    
    return {
        "Model": name,
        "R2_log": r2_log, "MAE_log": mae_log, "RMSE_log": rmse_log,
        "R2_actual": r2_actual, "MAE_actual": mae_actual, "RMSE_actual": rmse_actual
    }

In [36]:
from sklearn.svm import SVR

svr_C = 10           # try: 0.1, 1, 10, 100
svr_epsilon = 0.01     # try: 0.01, 0.1, 0.5
svr_kernel = 'rbf'    # try: 'linear', 'rbf', 'poly'

svr = SVR(C=svr_C, epsilon=svr_epsilon, kernel=svr_kernel)
res_svr = evaluate(f"SVR (C={svr_C}, eps={svr_epsilon}, kernel={svr_kernel})",
                    svr, x_train, y_train, x_test, y_test)

SVR (C=10, eps=0.01, kernel=rbf)
  [log scale]      R2: 0.3135  MAE: 0.7979  RMSE: 1.2180
  [minutes scale]  R2: 0.3090  MAE: 26.76 min  RMSE: 94.30 min



SVm is that good with the data so i am not considering it

In [37]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    max_iter=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
res_hgb = evaluate("HistGradientBoosting (iter=200, depth=5, lr=0.1)", hgb, x_train, y_train, x_test, y_test)

HistGradientBoosting (iter=200, depth=5, lr=0.1)
  [log scale]      R2: 0.6602  MAE: 0.5781  RMSE: 0.8569
  [minutes scale]  R2: 0.7677  MAE: 15.29 min  RMSE: 54.68 min



In [38]:
from sklearn.ensemble import HistGradientBoostingRegressor

hgb = HistGradientBoostingRegressor(
    max_iter=200,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
res_hgb = evaluate("HistGradientBoosting (iter=200, depth=5, lr=0.1)", hgb, x_train, y_train, x_test, y_test)

HistGradientBoosting (iter=200, depth=5, lr=0.1)
  [log scale]      R2: 0.6602  MAE: 0.5781  RMSE: 0.8569
  [minutes scale]  R2: 0.7677  MAE: 15.29 min  RMSE: 54.68 min



In [39]:
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
import numpy as np
import pandas as pd

# recombine train/test (already has failure_name_enc instead of raw failure_name)
x_combined = pd.concat([x_train, x_test]).sort_index()
y_combined = pd.concat([y_train, y_test]).sort_index()

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_cv(name, model, x, y):
    preds_log_cv = cross_val_predict(model, x, y, cv=kf)
    preds_actual = np.expm1(preds_log_cv)
    y_actual = np.expm1(y)
    r2 = r2_score(y_actual, preds_actual)
    mae = mean_absolute_error(y_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_actual, preds_actual))
    print(f"{name:40s}  R2={r2:.4f}  MAE={mae:6.2f} min  RMSE={rmse:6.2f} min")
    return {"Model": name, "R2": r2, "MAE": mae, "RMSE": rmse}

results = []

results.append(evaluate_cv("Linear Regression", LinearRegression(), x_combined, y_combined))
results.append(evaluate_cv("Ridge (alpha=1)", Ridge(alpha=1), x_combined, y_combined))
results.append(evaluate_cv("Random Forest (n=300, depth=None, leaf=1)",
    RandomForestRegressor(n_estimators=300, max_depth=None, min_samples_leaf=1, random_state=42), x_combined, y_combined))
results.append(evaluate_cv("Gradient Boosting (n=200, depth=2, lr=0.05)",
    GradientBoostingRegressor(n_estimators=200, max_depth=2, learning_rate=0.05, random_state=42), x_combined, y_combined))
results.append(evaluate_cv("AdaBoost (n=100, lr=0.1)",
    AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42), x_combined, y_combined))

comparison_df = pd.DataFrame(results)
comparison_df.sort_values("R2", ascending=False)

Linear Regression                         R2=0.3338  MAE= 23.03 min  RMSE= 82.61 min
Ridge (alpha=1)                           R2=0.3259  MAE= 23.13 min  RMSE= 83.10 min
Random Forest (n=300, depth=None, leaf=1)  R2=0.9584  MAE=  4.93 min  RMSE= 20.65 min
Gradient Boosting (n=200, depth=2, lr=0.05)  R2=0.8986  MAE=  9.78 min  RMSE= 32.23 min
AdaBoost (n=100, lr=0.1)                  R2=0.9678  MAE=  5.21 min  RMSE= 18.16 min


,Model,R2,MAE,RMSE
4,"AdaBoost (n=100, lr=0.1)",0.967807,5.208517,18.160669
2,"Random Forest (n=300, depth=None, leaf=1)",0.958374,4.934541,20.650580
3,"Gradient Boosting (n=200, depth=2, lr=0.05)",0.898586,9.784753,32.232834
0,Linear Regression,0.333846,23.025039,82.610979
1,Ridge (alpha=1),0.325906,23.129565,83.101893


In [40]:
from sklearn.model_selection import GridSearchCV, KFold, cross_val_predict
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import numpy as np
import pandas as pd

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def grid_search_and_eval(name, model, param_grid, x, y):
    grid = GridSearchCV(model, param_grid, cv=kf, scoring='r2', n_jobs=-1)
    grid.fit(x, y)

    best_model = model.__class__(**grid.best_params_, random_state=42) \
        if 'random_state' in model.get_params() else model.__class__(**grid.best_params_)

    preds_log_cv = cross_val_predict(best_model, x, y, cv=kf)
    preds_actual = np.expm1(preds_log_cv)
    y_actual = np.expm1(y)

    r2 = r2_score(y_actual, preds_actual)
    mae = mean_absolute_error(y_actual, preds_actual)
    rmse = np.sqrt(mean_squared_error(y_actual, preds_actual))

    print(f"{name}")
    print(f"  Best params: {grid.best_params_}")
    print(f"  [minutes] R2: {r2:.4f}  MAE: {mae:.2f} min  RMSE: {rmse:.2f} min\n")

    return {"Model": name, "Best Params": grid.best_params_, "R2": r2, "MAE": mae, "RMSE": rmse}

results = []

# Ridge — Linear Regression has no hyperparameters to tune, so it's skipped here (kept in your plain CV table instead)
results.append(grid_search_and_eval(
    "Ridge", Ridge(),
    {'alpha': [0.001, 0.01, 0.1, 1, 10, 100]},
    x_combined, y_combined
))

results.append(grid_search_and_eval(
    "Random Forest", RandomForestRegressor(random_state=42),
    {'n_estimators': [200, 300, 500], 'max_depth': [8, 10, 12, None], 'min_samples_leaf': [1, 2, 5]},
    x_combined, y_combined
))

results.append(grid_search_and_eval(
    "Gradient Boosting", GradientBoostingRegressor(random_state=42),
    {'n_estimators': [100, 200, 300], 'max_depth': [2, 3, 4], 'learning_rate': [0.01, 0.05, 0.1]},
    x_combined, y_combined
))

results.append(grid_search_and_eval(
    "AdaBoost", AdaBoostRegressor(random_state=42),
    {'n_estimators': [50, 100, 200], 'learning_rate': [0.01, 0.1, 1]},
    x_combined, y_combined
))

comparison_df = pd.DataFrame(results)
comparison_df.sort_values("R2", ascending=False)

Ridge
  Best params: {'alpha': 1}
  [minutes] R2: 0.3259  MAE: 23.13 min  RMSE: 83.10 min

Random Forest
  Best params: {'max_depth': 12, 'min_samples_leaf': 1, 'n_estimators': 300}
  [minutes] R2: 0.9584  MAE: 4.97 min  RMSE: 20.64 min

Gradient Boosting
  Best params: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 200}
  [minutes] R2: 0.9536  MAE: 6.34 min  RMSE: 21.81 min

AdaBoost
  Best params: {'learning_rate': 0.01, 'n_estimators': 100}
  [minutes] R2: 0.9676  MAE: 4.84 min  RMSE: 18.23 min



,Model,Best Params,R2,MAE,RMSE
3,AdaBoost,"{'learning_rate': 0.01, 'n_estimators': 100}",0.967563,4.842206,18.229271
1,Random Forest,"{'max_depth': 12, 'min_samples_leaf': 1, 'n_es...",0.958428,4.966998,20.637089
2,Gradient Boosting,"{'learning_rate': 0.1, 'max_depth': 3, 'n_esti...",0.953555,6.335427,21.813192
0,Ridge,{'alpha': 1},0.325906,23.129565,83.101893


In [41]:
decission_tree_model= DecisionTreeRegressor(max_depth=10, random_state=42)
evaluate_model(decission_tree_model, x_train, x_test, y_train, y_test, f"Decision Tree depth={depth}")
decission_tree_model.fit(x_train,y_train)

Decision Tree depth=10
MAE: 4.544098530251271
RMSE: 11.364595610214945
R2: 0.9899638597150447
----------------------------------------


,criterion,'squared_error'
,splitter,'best'
,max_depth,10
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,42
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,ccp_alpha,0.0


In [42]:
import pandas as pd

user_input = pd.DataFrame([{
    'mtbf_hours': 120,
    'month': 2,
    'hour': 14,
    'dayofweek': 3,
    'failure_name_enc': 25,

    'failure_category_Electrical': 0,
    'failure_category_Idle/External': 1,
    'failure_category_Logistics/Organization': 0,
    'failure_category_Mechanical': 0,
    'failure_category_Other': 0
}])

In [43]:
user_input[['mtbf_hours']] = scaler.transform(user_input[['mtbf_hours']])

In [44]:
import numpy as np

pred_log = decission_tree_model.predict(user_input)
pred_actual = np.expm1(pred_log)

print("Predicted impact (minutes):", pred_actual[0])

Predicted impact (minutes): 479.99999999999994


In [45]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def train_test_metrics(model, x_train, x_test, y_train, y_test):
    
    # Train model
    model.fit(x_train, y_train)
    
    # ======================
    # TRAINING PREDICTIONS
    # ======================
    y_train_pred_log = model.predict(x_train)
    
    y_train_actual = np.expm1(y_train)
    y_train_pred_actual = np.expm1(y_train_pred_log)
    
    train_mae = mean_absolute_error(y_train_actual, y_train_pred_actual)
    train_rmse = np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual))
    train_r2 = r2_score(y_train_actual, y_train_pred_actual)
    
    # ======================
    # TESTING PREDICTIONS
    # ======================
    y_test_pred_log = model.predict(x_test)
    
    y_test_actual = np.expm1(y_test)
    y_test_pred_actual = np.expm1(y_test_pred_log)
    
    test_mae = mean_absolute_error(y_test_actual, y_test_pred_actual)
    test_rmse = np.sqrt(mean_squared_error(y_test_actual, y_test_pred_actual))
    test_r2 = r2_score(y_test_actual, y_test_pred_actual)
    
    # ======================
    # PRINT RESULTS
    # ======================
    print("TRAINING METRICS")
    print("MAE:", train_mae)
    print("RMSE:", train_rmse)
    print("R2:", train_r2)
    print("-"*40)
    
    print("TESTING METRICS")
    print("MAE:", test_mae)
    print("RMSE:", test_rmse)
    print("R2:", test_r2)
    print("="*50)
    
    #return train_mae, train_rmse, train_r2, test_mae, test_rmse, test_r2

In [46]:
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(max_depth=10, random_state=42)

train_test_metrics(dt, x_train, x_test, y_train, y_test)

TRAINING METRICS
MAE: 2.2209790038699286
RMSE: 5.643472513751157
R2: 0.996672809579672
----------------------------------------
TESTING METRICS
MAE: 4.544098530251271
RMSE: 11.364595610214945
R2: 0.9899638597150447


In [47]:
from sklearn.ensemble import RandomForestRegressor

rf = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)

train_test_metrics(rf, x_train, x_test, y_train, y_test)

TRAINING METRICS
MAE: 4.495877110900097
RMSE: 17.851575421898726
R2: 0.9667081501822133
----------------------------------------
TESTING METRICS
MAE: 4.391359083092775
RMSE: 11.146671833058447
R2: 0.9903450688640378


In [48]:
from sklearn.ensemble import GradientBoostingRegressor

gbr = GradientBoostingRegressor(n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42)

train_test_metrics(gbr, x_train, x_test, y_train, y_test)

TRAINING METRICS
MAE: 3.078668226187106
RMSE: 8.588587483666316
R2: 0.992294012218204
----------------------------------------
TESTING METRICS
MAE: 5.438411096300039
RMSE: 14.886010725225525
R2: 0.9827807034715865


In [49]:
random_forest_model=RandomForestRegressor(max_depth=12,min_samples_leaf=1,n_estimators=300)
train_test_metrics(random_forest_model, x_train, x_test, y_train, y_test)

TRAINING METRICS
MAE: 3.3033312383127997
RMSE: 16.173858724148218
R2: 0.9726717312973271
----------------------------------------
TESTING METRICS
MAE: 4.22912303752373
RMSE: 10.683083770527002
R2: 0.9911314622780925


In [50]:
df.columns

Index(['event_id', 'timestamp', 'region', 'area', 'equipment', 'failure_name',
       'failure_category', 'severity', 'root_cause', 'root_cause_category',
       'impact_min', 'mttr_min', 'mtbf_hours', 'year', 'month'],
      dtype='object')

In [51]:
x_train.columns

Index(['mtbf_hours', 'month', 'hour', 'dayofweek', 'failure_name_enc',
       'failure_category_Electrical', 'failure_category_Idle/External',
       'failure_category_Logistics/Organization',
       'failure_category_Mechanical', 'failure_category_Other'],
      dtype='object')

In [52]:
x_combined.columns

Index(['mtbf_hours', 'month', 'hour', 'dayofweek', 'failure_name_enc',
       'failure_category_Electrical', 'failure_category_Idle/External',
       'failure_category_Logistics/Organization',
       'failure_category_Mechanical', 'failure_category_Other'],
      dtype='object')

In [53]:
name_to_enc = dict(zip(df['failure_name'], x_combined['failure_name_enc']))

In [58]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

# =========================
# MODEL
# =========================
rf = RandomForestRegressor(
    n_estimators=200,
    max_depth=8,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42
)

# =========================
# TRAIN
# =========================
rf.fit(x_train, y_train)

# =========================
# PREDICTIONS
# =========================
y_train_pred = rf.predict(x_train)
y_test_pred = rf.predict(x_test)

# =========================
# CONVERT BACK (log → actual)
# =========================
y_train_actual = np.expm1(y_train)
y_test_actual = np.expm1(y_test)

y_train_pred_actual = np.expm1(y_train_pred)
y_test_pred_actual = np.expm1(y_test_pred)

# =========================
# TRAINING METRICS
# =========================
train_mae = mean_absolute_error(y_train_actual, y_train_pred_actual)
train_rmse = np.sqrt(mean_squared_error(y_train_actual, y_train_pred_actual))
train_r2 = r2_score(y_train_actual, y_train_pred_actual)

print(" TRAINING METRICS")
print("MAE:", train_mae)
print("RMSE:", train_rmse)
print("R2:", train_r2)

# =========================
# TEST METRICS
# =========================
test_mae = mean_absolute_error(y_test_actual, y_test_pred_actual)
test_rmse = np.sqrt(mean_squared_error(y_test_actual, y_test_pred_actual))
test_r2 = r2_score(y_test_actual, y_test_pred_actual)

print("\n TEST METRICS")
print("MAE:", test_mae)
print("RMSE:", test_rmse)
print("R2:", test_r2)

 TRAINING METRICS
MAE: 6.066473514505085
RMSE: 25.82021403319957
R2: 0.9303526647786965

 TEST METRICS
MAE: 7.708663040059329
RMSE: 28.583291937792332
R2: 0.9365132288675149


In [ ]:
import joblib

# joblib.dump(decission_tree_model, "decision_tree_model.pkl")
# joblib.dump(x_combined.columns, "columns.pkl")
# joblib.dump(name_to_enc, "name_to_enc.pkl")
# joblib.dump(scaler, "scaler.pkl")  

In [61]:
import joblib
joblib.dump(rf, "random_forest_model.pkl")

['random_forest_model.pkl']